# Recovering Causal Effects from a Non-Random Feature Rollout
### A Causal Inference Case Study | Shlok Sheth

---

## Business Problem

A product team rolled out a new ad feature to a subset of users. The rollout was **not randomized**, high-engagement users were more likely to receive the feature. A naive analyst reports a **6.45% lift** in conversion rate and recommends full launch.

But is that lift real, or is it mostly explained by the fact that we targeted better users to begin with?

**This notebook answers that question using three complementary causal inference methods:**

| Method | Question Answered |
|--------|------------------|
| Propensity Score Matching (PSM) | What is the ATE after correcting for observed confounders? |
| Difference-in-Differences (DiD) | What is the causal lift controlling for pre-period baseline differences? |
| Uplift Modeling (T-Learner) | Which users actually respond to treatment — and which do not? |

---

**Dataset:** Criteo Uplift Modeling Dataset (synthetic replica, identical structure to v2.1)  
**Tools:** Python, scikit-learn, pandas, matplotlib  
**Author:** Shlok Sheth | sheth53@purdue.edu | https://www.linkedin.com/in/shlok-sheth2016/  
**Portfolio:** https://shethshlok.netlify.app

---
## 0. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import GradientBoostingClassifier
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0D1117',
    'axes.facecolor':   '#0D1117',
    'axes.edgecolor':   '#30363D',
    'axes.labelcolor':  '#C9D1D9',
    'xtick.color':      '#8B949E',
    'ytick.color':      '#8B949E',
    'text.color':       '#C9D1D9',
    'grid.color':       '#21262D',
    'grid.alpha':       0.6,
    'font.family':      'monospace',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

BLUE   = '#58A6FF'
ORANGE = '#F78166'
GREEN  = '#3FB950'
PURPLE = '#BC8CFF'
GOLD   = '#E3B341'
BG     = '#0D1117'
CARD   = '#161B22'

features = [f'f{i}' for i in range(12)]
print('Setup complete.')

---
## 1. Data Loading and Exploratory Analysis

Each row is one user. Key columns:

| Column | Description |
|--------|-------------|
| `f0` to `f11` | Anonymized user features (mix of continuous and binary) |
| `treatment` | 1 = received feature, 0 = control |
| `visit` | Pre-period engagement signal (proxy for baseline behavior) |
| `conversion` | Post-period outcome — did the user convert? |

> **To use the real Criteo dataset:** Download `criteo-uplift-v2.1.csv.gz` from https://ailab.criteo.com/criteo-uplift-modeling-dataset/ and replace the path below with `'../data/criteo-uplift-v2.1.csv.gz'`.

In [ ]:
df = pd.read_csv('../data/criteo_uplift_sample.csv')

print(f'Dataset shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nColumn types:')
print(df.dtypes)
df.head()

In [ ]:
n_total    = len(df)
n_treated  = df['treatment'].sum()
n_control  = n_total - n_treated
treat_rate = df[df['treatment']==1]['conversion'].mean()
ctrl_rate  = df[df['treatment']==0]['conversion'].mean()
naive_lift = treat_rate - ctrl_rate

print('=' * 55)
print(f'  Total users:              {n_total:,}')
print(f'  Treated:                  {n_treated:,} ({n_treated/n_total:.1%})')
print(f'  Control:                  {n_control:,} ({n_control/n_total:.1%})')
print('=' * 55)
print(f'  Treated conversion rate:  {treat_rate:.4f}')
print(f'  Control conversion rate:  {ctrl_rate:.4f}')
print(f'  Naive lift:               {naive_lift:.4f}')
print('=' * 55)
print(f'\n  WARNING: Naive lift of {naive_lift:.4f} is biased.')
print('  Treatment was not randomly assigned — high-engagement')
print('  users were more likely to receive the feature.')
print('  Causal inference methods are required.')

---
## 2. Diagnosing Selection Bias

We quantify imbalance using **standardized mean differences (SMD)** for each feature.

**Rule of thumb:** SMD > 0.10 indicates meaningful imbalance between treated and control groups that will bias a naive comparison.

In [ ]:
t_means    = df[df['treatment']==1][features].mean()
c_means    = df[df['treatment']==0][features].mean()
pooled_std = df[features].std()
smd        = ((t_means - c_means) / pooled_std).abs().sort_values(ascending=False)

imbalance_df = pd.DataFrame({
    'Treated Mean':    t_means,
    'Control Mean':    c_means,
    'Abs SMD':         smd,
    'Biased (>0.10)':  smd > 0.10
}).sort_values('Abs SMD', ascending=False)

print(imbalance_df.round(4).to_string())
print(f'\nFeatures with significant imbalance: {(smd > 0.10).sum()} of {len(features)}')

In [ ]:
smd_sorted = smd.sort_values(ascending=True)
colors_smd = [ORANGE if v > 0.10 else GREEN for v in smd_sorted]

fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG)
ax.set_facecolor(BG)
ax.barh(smd_sorted.index, smd_sorted.values, color=colors_smd, height=0.6, zorder=3)
ax.axvline(0.10, color=GOLD, linestyle='--', linewidth=1.5, label='Bias threshold (SMD = 0.10)', zorder=4)
ax.set_xlabel('Absolute Standardized Mean Difference', fontsize=11)
ax.set_title(
    'Covariate Imbalance Before Matching\n'
    'Features above the threshold confirm non-random treatment assignment',
    fontsize=12, fontweight='bold', pad=15, color='#E6EDF3'
)
ax.legend(fontsize=9, facecolor=CARD, edgecolor='#30363D')
ax.grid(axis='x', zorder=0)
biased_n = (smd > 0.10).sum()
ax.text(0.98, 0.04,
        f'{biased_n} of {len(features)} features show significant imbalance',
        transform=ax.transAxes, ha='right', fontsize=9, color=ORANGE,
        bbox=dict(boxstyle='round,pad=0.4', facecolor=CARD, edgecolor=ORANGE, alpha=0.9))
plt.tight_layout(pad=2)
plt.show()

---
## 3. Method 1 — Propensity Score Matching (PSM)

**Core idea:** Match each treated user to a statistically similar control user based on their probability of being treated (the propensity score). After matching, the two groups are comparable, and the difference in outcomes estimates the Average Treatment Effect (ATE).

**Steps:**
1. Estimate propensity scores via logistic regression
2. Verify common support (distributional overlap)
3. Match 1:1 nearest neighbor on propensity score
4. Verify balance improved post-matching
5. Compute ATE with bootstrap confidence interval

In [ ]:
X      = df[features]
y      = df['treatment']
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_sc, y)
df['ps'] = lr.predict_proba(X_sc)[:, 1]

print(f'Logistic Regression trained.')
print(f'PS range: [{df["ps"].min():.4f}, {df["ps"].max():.4f}]')
print(f'Mean PS (treated): {df[df["treatment"]==1]["ps"].mean():.4f}')
print(f'Mean PS (control): {df[df["treatment"]==0]["ps"].mean():.4f}')
print(f'\nHigh mean PS difference confirms selection bias is present.')

In [ ]:
treated_ps  = df[df['treatment']==1]['ps']
control_ps  = df[df['treatment']==0]['ps']
overlap_min = max(treated_ps.min(), control_ps.min())
overlap_max = min(treated_ps.max(), control_ps.max())
overlap_pct = ((df['ps'] >= overlap_min) & (df['ps'] <= overlap_max)).mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor=BG)
for ax in axes:
    ax.set_facecolor(BG)

axes[0].hist(treated_ps, bins=60, alpha=0.7, color=BLUE,   label='Treated', density=True, zorder=3)
axes[0].hist(control_ps, bins=60, alpha=0.7, color=ORANGE, label='Control', density=True, zorder=3)
axes[0].set_xlabel('Propensity Score', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Propensity Score Distribution', fontsize=11, fontweight='bold', pad=12, color='#E6EDF3')
axes[0].legend(fontsize=10, facecolor=CARD, edgecolor='#30363D')
axes[0].grid(axis='y', zorder=0)

axes[1].hist(treated_ps, bins=60, alpha=0.7, color=BLUE,   label='Treated', density=True, zorder=3)
axes[1].hist(control_ps, bins=60, alpha=0.7, color=ORANGE, label='Control', density=True, zorder=3)
axes[1].axvspan(overlap_min, overlap_max, alpha=0.12, color=GREEN, zorder=2,
                label=f'Common support ({overlap_pct:.1f}%)')
axes[1].set_xlabel('Propensity Score', fontsize=11)
axes[1].set_title('Common Support Region', fontsize=11, fontweight='bold', pad=12, color='#E6EDF3')
axes[1].legend(fontsize=9, facecolor=CARD, edgecolor='#30363D')
axes[1].grid(axis='y', zorder=0)

plt.suptitle('Propensity Score Estimation via Logistic Regression',
             fontsize=13, fontweight='bold', color='#E6EDF3', y=1.02)
plt.tight_layout(pad=2)
plt.show()

print(f'Common support: {overlap_pct:.1f}% of users fall in the overlap region.')

In [ ]:
sample_t = df[df['treatment']==1].sample(5000, random_state=42).copy()
sample_c = df[df['treatment']==0].sample(20000, random_state=42).copy()

nn = NearestNeighbors(n_neighbors=1, algorithm='ball_tree')
nn.fit(sample_c[['ps']])
distances, indices = nn.kneighbors(sample_t[['ps']])

matched_c = sample_c.iloc[indices.flatten()].copy()
matched_t = sample_t.copy()

print(f'Matched pairs: {len(matched_t):,}')
print(f'Max PS distance:  {distances.max():.5f}')
print(f'Mean PS distance: {distances.mean():.5f}')

In [ ]:
t_after    = matched_t[features].mean()
c_after    = matched_c[features].mean()
smd_after  = ((t_after - c_after) / df[features].std()).abs()
avg_before = smd.reindex(features).mean()
avg_after  = smd_after.reindex(features).mean()

fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
ax.set_facecolor(BG)
x = np.arange(len(features))
w = 0.35
ax.bar(x - w/2, smd.reindex(features).values, w, color=ORANGE, alpha=0.85, label='Before Matching', zorder=3)
ax.bar(x + w/2, smd_after.reindex(features).values, w, color=GREEN, alpha=0.85, label='After Matching', zorder=3)
ax.axhline(0.10, color=GOLD, linestyle='--', linewidth=1.5, label='Bias threshold (0.10)', zorder=4)
ax.set_xticks(x)
ax.set_xticklabels(features, fontsize=9)
ax.set_ylabel('Absolute Standardized Mean Difference', fontsize=11)
ax.set_title(
    'Covariate Balance: Before vs After Propensity Score Matching\n'
    'Matching brings all features below the 0.10 bias threshold',
    fontsize=11, fontweight='bold', pad=15, color='#E6EDF3'
)
ax.legend(fontsize=10, facecolor=CARD, edgecolor='#30363D')
ax.grid(axis='y', zorder=0)
ax.text(0.02, 0.96,
        f'Avg SMD before: {avg_before:.3f}  →  after: {avg_after:.3f}',
        transform=ax.transAxes, va='top', fontsize=9, color=GREEN,
        bbox=dict(boxstyle='round,pad=0.4', facecolor=CARD, edgecolor=GREEN, alpha=0.9))
plt.tight_layout(pad=2)
plt.show()

In [ ]:
psm_ate  = (matched_t['conversion'].values - matched_c['conversion'].values).mean()

n_boot, n_pairs = 1000, len(matched_t)
boot_ates = []
for _ in range(n_boot):
    idx  = np.random.choice(n_pairs, n_pairs, replace=True)
    diff = matched_t['conversion'].values[idx] - matched_c['conversion'].values[idx]
    boot_ates.append(diff.mean())
ci_lo = np.percentile(boot_ates, 2.5)
ci_hi = np.percentile(boot_ates, 97.5)

print('=' * 55)
print(f'  Naive lift (biased):       {naive_lift:.4f}')
print(f'  PSM ATE (corrected):       {psm_ate:.4f}')
print(f'  95% Bootstrap CI:          [{ci_lo:.4f}, {ci_hi:.4f}]')
print(f'  Bias removed:              {(naive_lift - psm_ate)/naive_lift*100:.1f}%')
print('=' * 55)

---
## 4. Method 2 — Difference-in-Differences (DiD)

**Core idea:** Compare the *change* in outcomes between treated and control groups across two time periods. If both groups would have followed the same trend without treatment (parallel trends assumption), the DiD estimate is causal.

**Setup:**
- **Pre-period proxy:** `visit` — measures baseline engagement before treatment
- **Post-period:** `conversion` — the outcome of interest
- **DiD = (Post_treated − Pre_treated) − (Post_control − Pre_control)**

In [ ]:
pre_treat    = df[df['treatment']==1]['visit'].mean()
pre_control  = df[df['treatment']==0]['visit'].mean()
post_treat   = df[df['treatment']==1]['conversion'].mean()
post_control = df[df['treatment']==0]['conversion'].mean()

did_estimate       = (post_treat - pre_treat) - (post_control - pre_control)
counterfactual_post = pre_treat + (post_control - pre_control)

print('Difference-in-Differences Table')
print('=' * 55)
print(f'                       Treated    Control    Diff')
print(f'  Pre-period  (visit)  {pre_treat:.4f}     {pre_control:.4f}    {pre_treat - pre_control:+.4f}')
print(f'  Post-period (conv)   {post_treat:.4f}     {post_control:.4f}    {post_treat - post_control:+.4f}')
print('=' * 55)
print(f'  DiD Estimate:                          {did_estimate:+.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor=BG)
for ax in axes:
    ax.set_facecolor(BG)

periods = ['Pre-Period\n(Visit Rate)', 'Post-Period\n(Conversion Rate)']

ax0 = axes[0]
ax0.plot(periods, [pre_treat,  post_treat],         'o-',  color=BLUE,   linewidth=2.5, markersize=9, label='Treated',                   zorder=5)
ax0.plot(periods, [pre_control, post_control],       'o-',  color=ORANGE, linewidth=2.5, markersize=9, label='Control',                   zorder=5)
ax0.plot(periods, [pre_treat, counterfactual_post],  'o--', color=BLUE,   linewidth=1.5, markersize=9, alpha=0.4, label='Counterfactual', zorder=4)
ax0.fill_between([1], [counterfactual_post], [post_treat], alpha=0.2, color=GREEN, zorder=3)
ax0.set_ylabel('Rate', fontsize=11)
ax0.set_title('Difference-in-Differences\nParallel Trends Visualization', fontsize=11, fontweight='bold', pad=12, color='#E6EDF3')
ax0.legend(fontsize=9, facecolor=CARD, edgecolor='#30363D')
ax0.grid(axis='y', zorder=0)
ax0.text(1.05, (counterfactual_post + post_treat)/2,
         f'DiD = {did_estimate:.4f}', va='center', fontsize=10, color=GREEN, fontweight='bold')

ax1 = axes[1]
methods_cmp = ['Naive\nComparison', 'PSM\nATE', 'DiD\nEstimate']
values_cmp  = [naive_lift, psm_ate, did_estimate]
colors_cmp  = [ORANGE, BLUE, PURPLE]
bars = ax1.bar(methods_cmp, values_cmp, color=colors_cmp, width=0.5, zorder=3, alpha=0.9)
ax1.axhline(0.0142, color=GOLD, linestyle='--', linewidth=2, label='Ground truth ATE (0.0142)', zorder=4)
for bar, val in zip(bars, values_cmp):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{val:.4f}', ha='center', fontsize=11, fontweight='bold', color='#E6EDF3')
ax1.set_ylabel('Estimated Treatment Effect', fontsize=11)
ax1.set_title('Method Comparison', fontsize=11, fontweight='bold', pad=12, color='#E6EDF3')
ax1.legend(fontsize=9, facecolor=CARD, edgecolor='#30363D')
ax1.grid(axis='y', zorder=0)
ax1.set_ylim(0, max(values_cmp) * 1.25)

plt.suptitle('Difference-in-Differences and Method Comparison',
             fontsize=13, fontweight='bold', color='#E6EDF3', y=1.02)
plt.tight_layout(pad=2)
plt.show()

---
## 5. Method 3 — Uplift Modeling (T-Learner)

PSM and DiD estimate a single **Average Treatment Effect** across the whole population. The more actionable product question is:

> **Which users actually respond to the treatment — and which are unaffected or harmed?**

The **T-Learner** (Two-Model approach) estimates individual treatment effects:
1. Train model on **treated users only** → `P(conversion | X, T=1)`
2. Train model on **control users only** → `P(conversion | X, T=0)`
3. Individual uplift = `model_treated.predict(X) − model_control.predict(X)`

**Segment interpretation:**
- **Top 25% (Sure Things / Persuadables):** Target these users — highest causal lift
- **Bottom 25% (Sleeping Dogs / Lost Causes):** Avoid these users — zero or negative lift

In [ ]:
sample = df.sample(30000, random_state=42).copy()

treated_data = sample[sample['treatment']==1]
control_data = sample[sample['treatment']==0]

model_t = GradientBoostingClassifier(n_estimators=80, max_depth=4, random_state=42)
model_c = GradientBoostingClassifier(n_estimators=80, max_depth=4, random_state=42)

print('Training treated model...')
model_t.fit(treated_data[features], treated_data['conversion'])
print('Training control model...')
model_c.fit(control_data[features], control_data['conversion'])

p1 = model_t.predict_proba(sample[features])[:, 1]
p0 = model_c.predict_proba(sample[features])[:, 1]
sample['uplift_score'] = p1 - p0

print(f'\nUplift score summary:')
print(sample['uplift_score'].describe().round(4))

In [ ]:
sample['uplift_segment'] = pd.qcut(
    sample['uplift_score'], q=4,
    labels=['Bottom 25%\n(Lost Causes)', 'Q2\n(Low Responders)',
            'Q3\n(Persuadables)', 'Top 25%\n(Sure Things)']
)

seg_summary = sample.groupby('uplift_segment', observed=True).agg(
    n=('conversion', 'count'),
    conversion_treated=('conversion', lambda x: x[sample.loc[x.index, 'treatment']==1].mean()),
    conversion_control=('conversion',  lambda x: x[sample.loc[x.index, 'treatment']==0].mean()),
    avg_uplift=('uplift_score', 'mean')
).reset_index()
seg_summary['observed_lift'] = seg_summary['conversion_treated'] - seg_summary['conversion_control']

print('Uplift Segment Summary')
print('=' * 75)
print(seg_summary[['uplift_segment', 'n', 'conversion_treated',
                   'conversion_control', 'observed_lift', 'avg_uplift']].to_string(index=False))
print('=' * 75)
top_lift = seg_summary.iloc[-1]['observed_lift']
print(f'\nTop segment lift:    {top_lift:.4f}')
print(f'Bottom segment lift: {seg_summary.iloc[0]["observed_lift"]:.4f} (negative — avoid targeting these users)')
print(f'\nTargeting only the top 25% yields {top_lift/naive_lift:.1f}x the naive population lift.')

In [ ]:
seg_colors = [ORANGE, GOLD, BLUE, GREEN]
seg_labels = ['Bottom 25%\n(Lost Causes)', 'Q2\n(Low Responders)',
              'Q3\n(Persuadables)', 'Top 25%\n(Sure Things)']

fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor=BG)
for ax in axes:
    ax.set_facecolor(BG)

ax0 = axes[0]
ax0.bar(seg_labels, seg_summary['avg_uplift'], color=seg_colors, width=0.6, zorder=3, alpha=0.9)
ax0.axhline(0, color='#8B949E', linewidth=1, zorder=2)
ax0.set_ylabel('Average Predicted Uplift Score', fontsize=11)
ax0.set_title('Predicted Uplift by Segment\n(T-Learner Two-Model Approach)', fontsize=11, fontweight='bold', pad=12, color='#E6EDF3')
ax0.grid(axis='y', zorder=0)
for i, val in enumerate(seg_summary['avg_uplift']):
    ax0.text(i, val + (0.003 if val >= 0 else -0.006),
             f'{val:.4f}', ha='center', fontsize=10, fontweight='bold', color='#E6EDF3')

ax1 = axes[1]
x_pos = np.arange(len(seg_labels))
w     = 0.35
ax1.bar(x_pos - w/2, seg_summary['conversion_treated'], w, color=BLUE,   alpha=0.85, label='Treated', zorder=3)
ax1.bar(x_pos + w/2, seg_summary['conversion_control'],  w, color=ORANGE, alpha=0.85, label='Control', zorder=3)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(seg_labels, fontsize=9)
ax1.set_ylabel('Conversion Rate', fontsize=11)
ax1.set_title('Observed Conversion Rate by Segment\nTreated vs Control Within Each Bucket', fontsize=11, fontweight='bold', pad=12, color='#E6EDF3')
ax1.legend(fontsize=10, facecolor=CARD, edgecolor='#30363D')
ax1.grid(axis='y', zorder=0)

plt.suptitle('Uplift Modeling: Who Actually Responds to Treatment?',
             fontsize=13, fontweight='bold', color='#E6EDF3', y=1.02)
plt.tight_layout(pad=2)
plt.show()

In [ ]:
sample_sorted = sample.sort_values('uplift_score', ascending=False).reset_index(drop=True)
n_s           = len(sample_sorted)
fracs         = np.linspace(0.01, 1.0, 100)

qini_vals = []
for frac in fracs:
    k   = max(1, int(frac * n_s))
    top = sample_sorted.iloc[:k]
    n_t = (top['treatment'] == 1).sum()
    n_c = (top['treatment'] == 0).sum()
    if n_t > 0 and n_c > 0:
        q = (top[top['treatment']==1]['conversion'].sum() / n_t -
             top[top['treatment']==0]['conversion'].sum() / n_c)
    else:
        q = 0
    qini_vals.append(q)

fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG)
ax.set_facecolor(BG)
ax.plot(fracs * 100, qini_vals, color=BLUE, linewidth=2.5, label='Uplift Model', zorder=5)
ax.axhline(naive_lift, color=ORANGE, linewidth=1.5, linestyle='--',
           label=f'Random targeting (naive lift = {naive_lift:.4f})', zorder=4)
ax.axhline(0, color='#8B949E', linewidth=1, zorder=3)
ax.fill_between(fracs * 100, naive_lift, qini_vals,
                where=[q > naive_lift for q in qini_vals],
                alpha=0.15, color=GREEN, label='Model advantage over random targeting')
ax.set_xlabel('Percentage of Population Targeted (by uplift score, descending)', fontsize=11)
ax.set_ylabel('Observed Lift (Treated Rate minus Control Rate)', fontsize=11)
ax.set_title(
    'Qini-Style Cumulative Gain Curve\n'
    'Top-scored users show the highest causal lift — model beats random targeting',
    fontsize=11, fontweight='bold', pad=15, color='#E6EDF3'
)
ax.legend(fontsize=10, facecolor=CARD, edgecolor='#30363D')
ax.grid(zorder=0)
plt.tight_layout(pad=2)
plt.show()

---
## 6. Results Summary and Business Recommendation

In [ ]:
print('=' * 70)
print('  CAUSAL INFERENCE RESULTS SUMMARY')
print('=' * 70)
print(f'  {"Method":<35} {"Estimate":>10}  Notes')
print('-' * 70)
print(f'  {"Naive comparison (biased)":<35} {naive_lift:>10.4f}  Selection bias present')
print(f'  {"PSM ATE (corrected)":<35} {psm_ate:>10.4f}  Observed confounders removed')
print(f'  {"Difference-in-Differences":<35} {did_estimate:>10.4f}  Baseline differences controlled')
print(f'  {"Ground Truth ATE (synthetic)":<35} {0.0142:>10.4f}  Known from data generation')
print('=' * 70)
print(f'\n  Naive overstatement:      {(naive_lift - psm_ate)/naive_lift*100:.1f}%')
print(f'  PSM 95% CI:               [{ci_lo:.4f}, {ci_hi:.4f}]')
print(f'  Top 25% segment lift:     {seg_summary.iloc[-1]["observed_lift"]:.4f}')
print(f'  Bottom 25% segment lift:  {seg_summary.iloc[0]["observed_lift"]:.4f} (negative)')
print('=' * 70)

---
## Business Recommendation

### Do not ask "Did it work?" — ask "For whom, and at what cost?"

| Finding | Implication |
|---------|-------------|
| Naive lift of **0.0645** is biased by **85.7%** | Do not use this estimate to justify a full launch |
| PSM ATE of **0.0092** is the corrected estimate | True population-average lift is far smaller than reported |
| Top 25% uplift segment shows roughly **5x** average lift | Targeted rollout is dramatically more efficient |
| Bottom 25% shows **negative** observed lift | Broad rollout actively harms conversion for some users |

**Recommended action:** Restrict the feature rollout to users in the **top uplift quartile**. This concentrates resources on Persuadables, avoids harming Sleeping Dogs, and maximizes ROI per treated user.

**Validation path:** Run a follow-up RCT restricted to high-uplift-score users to confirm individual treatment effect (ITE) estimates before a permanent launch decision.

---

## Limitations

1. **PSM** corrects only for *observed* confounders. Unmeasured confounding (device type, geography, session depth) may remain and bias the estimate.
2. **DiD** relies on the parallel trends assumption, which cannot be directly tested and can only be made plausible with additional pre-period data.
3. **Uplift model** was trained on observational data. ITE estimates should be treated as directional signals, not causal facts, until validated via a controlled experiment.
4. **Scale:** This notebook uses a 100K-row synthetic sample that mirrors Criteo v2.1 structure. The full dataset contains 14 million rows and may produce different estimates.

---

*Shlok Sheth | Purdue University | sheth53@purdue.edu | https://shethshlok.netlify.app*